In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [5]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.item_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lý null

### Xử lý age_group

In [6]:
df_age = read_parquet_item("./preprocessed-dataset")
df_age.head()

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final,age_group_from_desc_str,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""","""9M+""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định""","""Bé Gái""",null,"""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""",null,"""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null,"""12-36M"""


#### Ý tưởng 2 (tiếp theo):

##### Thống kê

In [7]:
import polars as pl

df = df_age   # hoặc df_filled tùy bạn đang dùng

# Hàm tiện dụng để lấy danh sách unique của 1 cột
def get_unique_list(df, col):
    return (
        df.select(col)
          .unique()
          .sort(col)
          .get_column(col)
          .to_list()
    )

# Lấy tất cả class
age_list    = get_unique_list(df, "age_group_final")

# Gom vào dictionary
category_dict = {
    "age_group": age_list
}

print(len(age_list))

# In ra theo từng dòng, rất dễ đọc
print("\n=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===\n")
for key, value in category_dict.items():
    print(f"{key}: {value}\n")

311

=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===

age_group: ['0-10M', '0-11M', '0-120M', '0-12M', '0-144M', '0-14M', '0-18M', '0-1M', '0-24M', '0-2M', '0-30M', '0-36M', '0-3M', '0-48M', '0-4M', '0-5M', '0-60M', '0-6M', '0-72M', '0-84M', '0-9M', '1-12M', '1-18M', '1-24M', '1-36M', '1-3M', '108-120M', '108-132M', '12-108M', '12-120M', '12-132M', '12-144M', '12-14M', '12-192M', '12-24M', '12-36M', '12-48M', '12-60M', '12-72M', '120-144M', '12M-18M', '12M-4Y', '13-16M', '13-17M', '132-144M', '13M-24M', '14-17M', '18-24M', '18M-24M', '18M-36M', '18M-4Y', '1M-12M', '1M-15M', '1M-3M', '24-120M', '24-36M', '24-48M', '24-60M', '24-72M', '24-96M', '2M-15M', '2M-6M', '3-18M', '3-24M', '3-6M', '36-120M', '36-144M', '36-216M', '36-48M', '36-60M', '36-72M', '3M-12M', '3M-18M', '3M-24M', '3M-6M', '4-24M', '4-30M', '4-6M', '48-60M', '48-72M', '4M-4Y', '4M-6M', '6-12M', '6-144M', '6-15M', '6-18M', '6-24M', '6-30M', '6-36M', '6-60M', '6-72M', '6-84M', '6-8M', '6-9M', '60-72M', '6M-10M', '6M-12M', 

In [8]:
import polars as pl
import textwrap

# ===============================
# 1) LỌC DỮ LIỆU CẦN XUẤT
# ===============================
import polars as pl

# Điều kiện "có thông tin" cho description
desc_has_info = (
    pl.col("description").is_not_null()
    & (pl.col("description") != "Không xác định")
    & (pl.col("description").str.strip_chars() != "")
)

# Điều kiện "có thông tin" cho description_new
desc_new_has_info = (
    pl.col("description_new").is_not_null()
    & (pl.col("description_new") != "Không xác định")
    & (pl.col("description_new").str.strip_chars() != "")
)

# Lọc các dòng cần thống kê
df_unknown_desc = (
    df_age
    .filter(
        (pl.col("age_group_final") == "Không xác định")
        & (desc_has_info | desc_new_has_info)
    )
    .select([
        "item_id",
        "description",
        "description_new",
        "age_group_final",
    ])
)
print("Số dòng cần xuất:", df_unknown_desc.height)

# ===============================
# 2) HÀM FORMAT TEXT
# ===============================
def wrap175(text):
    if text is None:
        return ""
    return textwrap.fill(str(text), width=175)

# ===============================
# 3) XUẤT RA FILE TXT
# ===============================
output_file = "unknown_description_items.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for row in df_unknown_desc.iter_rows(named=True):
        f.write("=====================================\n")
        f.write(f"item_id: {row['item_id']}\n\n")

        f.write("age_group_final: ")
        f.write(f"{row['age_group_final']}\n\n")

        f.write("description:\n")
        f.write(wrap175(row["description"]) + "\n\n")

        f.write("description_new:\n")
        f.write(wrap175(row["description_new"]) + "\n\n")

print(f"Đã xuất file: {output_file}")


Số dòng cần xuất: 3792
Đã xuất file: unknown_description_items.txt


##### Thử fill lại

In [9]:
import re
import textwrap
from typing import Optional, List, Tuple

import polars as pl

# =========================================
# 1. CẤU HÌNH CONTEXT & TỪ KHÓA
# =========================================

POS_CONTEXT = [
    # Trẻ em / baby
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
    "thiếu nhi", "newborn", "sơ sinh",

    # Ngữ cảnh độ tuổi
    "độ tuổi",
    "tháng tuổi",
    "months old",
    "years old",
    "tuổi",
]

CHILD_WORDS = [
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
]

NEG_STRONG = [
    # hạn sử dụng / bảo quản
    "sử dụng trong vòng",
    "hạn sử dụng", "thời hạn", "hạn dùng", "hạn sử dụng tốt nhất",
    "bảo quản", "kể từ ngày sản xuất", "sau khi mở nắp",

    # không dùng / không phù hợp cho trẻ
    "không thích hợp cho trẻ",
    "không thích hợp với trẻ",
    "không dùng cho trẻ",
    "không nên dùng cho trẻ",
    "không phù hợp cho trẻ",
    "không phù hợp với trẻ",
    "không sử dụng sản phẩm cho trẻ",
    "không sử dụng cho trẻ",

    # tiếng Anh
    "not suitable for children",
    "do not use for children",
]

# Các từ mô tả cân nặng / thể tích / kích thước (ngữ cảnh đo lường)
MEASURE_WORDS = [
    "kg", "kilogram", "ký", "kí",
    "cân", "nặng", "trọng lượng", "weight",
    "ml", "lít", "lit", "cc",
    "cm", "mm", "m²", "m2",
]

# Đơn vị thời gian dùng để quy đổi -> THÁNG
MONTH_WORDS = ["tháng", "thang", "month", "months"]
YEAR_WORDS  = ["tuổi", "year", "years", "y"]
WEEK_WORDS  = ["tuần", "tuan", "week", "weeks"]

WINDOW_POS = 20
WINDOW_NEG = 80

# Các cụm nói về "dưới sự hướng dẫn/giám sát"
NEG_SUPERVISION = [
    "dưới sự hướng dẫn",
    "cần sử dụng dưới sự hướng dẫn",
    "dưới sự giám sát",
    "under supervision",
    "under the guidance",
]


# =========================================
# 2. PREPROCESS TEXT TRƯỚC KHI MATCH
# =========================================

def preprocess_text_for_age(text: Optional[str]) -> str:
    """
    Chuẩn hoá text:
      - Nếu None -> ""
      - Chuẩn hoá khoảng trắng
      - Chèn space chỗ dính chữ: 'hợpBé' -> 'hợp Bé'
      - Chèn space trước '(' nếu dính
      - Chèn space sau ':' nếu thiếu
    """
    if text is None:
        return ""

    t = str(text)

    # Chuẩn hoá khoảng trắng
    t = re.sub(r"\s+", " ", t)

    # Tách giữa 1 ký tự bất kỳ (không phải space) và chữ hoa (kể cả Đ)
    # 'hợpBé' -> 'hợp Bé', 'gáiĐộ' -> 'gái Độ'
    t = re.sub(r"([^\s])([A-ZĐ])", r"\1 \2", t)

    # 'Formula(1-3 tuổi)' -> 'Formula (1-3 tuổi)'
    t = re.sub(r"([a-zA-ZÀ-ỹ])\(", r"\1 (", t)

    # 'Độ tuổi:5-7Y' -> 'Độ tuổi: 5-7Y'
    t = re.sub(r":([^\s])", r": \1", t)

    return t.strip()

In [10]:
# =========================================
# 3. REGEX PATTERNS CHO EXTRACT
# =========================================

# 3.0. Range đặc biệt kiểu "3M-5T" (tháng - tuổi)
AGE_MT_RANGE_PATTERN = re.compile(
    r"(?P<n1>\d{1,2})\s*m\s*(?:-|–|—|đến|to)\s*(?P<n2>\d{1,2})\s*t"
)

# 3.1. Range kiểu "0-6 tháng ... đến ... 3 tuổi"
CROSS_MONTH_RANGE_TO_YEAR_PATTERN = re.compile(
    r"(?P<m_start>\d{1,2})\s*-\s*(?P<m_end>\d{1,2})\s*"
    r"(?:tháng|thang|month|months)"
    r".{0,40}?(?:đến|tới|to)\s*.{0,40}?"
    r"(?P<y_end>\d{1,2}(?:[.,]\d+)?)\s*(?:tuổi|year|years|y)",
)

# 3.2. Newborn range "sơ sinh ... đến ... 3 tuổi"
NEWBORN_RANGE_PATTERN = re.compile(
    r"(?:từ\s+giai\s*đoạn\s+)?sơ\s*sinh"
    r".{0,30}?(?:đến|tới|to|-).{0,30}?"
    r"\d{1,2}(?:[.,]\d+)?\s*(tháng|thang|month|months|tuổi|year|years|y)",
)

# 3.3. "độ tuổi từ 2 đến 6"
AGE_WORD_RANGE_PATTERN = re.compile(
    r"độ\s*tuổi[^0-9]{0,40}"           # cho phép nhiều chữ hơn (hợpBé, phù hợp cho bé, ...)
    r"(?:từ\s*)?"
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)"
)

# 3.4. Range 2 đơn vị: "6 tháng đến 5 tuổi", "từ 1 tuần - 1 tuổi"
RANGE_BOTH_UNITS_PATTERN = re.compile(
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u1>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
    r"(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u2>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
)

# 3.5. Range 1 đơn vị: "0-12 tháng", "2-6 tuổi"
RANGE_ONE_UNIT_PATTERN = re.compile(
    r"(?:(?:từ)\s*)?"
    r"(?P<n1>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?:-|–|—|đến|to)\s*"
    r"(?P<n2>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
)

# 3.6. Single: "dưới 12 tháng", "từ 6 tháng tuổi", "2.5 tuổi"
SINGLE_PATTERN = re.compile(
    r"(?:(?P<cmp>dưới|under|<|từ|trên|hơn|sau|>=|over|more than)\s*)?"
    r"(?P<n>\d{1,2}(?:[.,]\d+)?)\s*"
    r"(?P<u>tháng tuổi|tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
    r"(?:\s*(?P<suffix>trở lên|\+))?"
)

In [11]:
# =========================================
# 4. HÀM CONTEXT POS / NEG
# =========================================

def has_pos_context(context: str) -> bool:
    return any(tok in context for tok in POS_CONTEXT)


def has_neg_context(context: str) -> bool:
    ctx = context

    # 1) NEG mạnh (hạn sử dụng / bảo quản / sau khi mở nắp...)
    if any(phrase in ctx for phrase in NEG_STRONG):
        return True

    # 2) "không ... (phù hợp/thích hợp/dùng/sử dụng/bôi/cho) ... trẻ"
    if "không" in ctx:
        if any(v in ctx for v in ["phù hợp", "thích hợp", "dùng", "sử dụng", "bôi", "cho"]):
            if any(c in ctx for c in CHILD_WORDS):
                return True

    # 3) "tránh ... trẻ" (tránh cho trẻ em dưới 1 tuổi...)
    if "tránh" in ctx and any(c in ctx for c in CHILD_WORDS):
        return True

    # 4) English NEG
    if "not suitable" in ctx and any(c in ctx for c in ["child", "children", "kid", "baby"]):
        return True
    if "do not use" in ctx and any(c in ctx for c in ["child", "children", "kid", "baby"]):
        return True

    # 5) CASE "trẻ dưới X tuổi cần sử dụng dưới sự hướng dẫn..."
    if any(phrase in ctx for phrase in NEG_SUPERVISION):
        if "dưới" in ctx or "under" in ctx:
            if any(c in ctx for c in CHILD_WORDS):
                return True

    # 6) CASE THỜI LƯỢNG "6 tháng đầu", "trong 6 tháng đầu"
    if "tháng đầu" in ctx or "tháng đầu đời" in ctx:
        if any(k in ctx for k in ["trong ", "trong vòng", "giai đoạn"]):
            return True

    # 7) CASE CÂN NẶNG / THỂ TÍCH / KÍCH THƯỚC
    if any(w in ctx for w in MEASURE_WORDS):
        return True

    return False

In [12]:
# =========================================
# 5. HÀM EXTRACT_AGE_PHRASES (DÙNG desc_all)
# =========================================

def extract_age_phrases(desc_all: Optional[str]) -> List[str]:
    """
    Nhận 1 chuỗi desc_all (merge 2 description),
    preprocess và trả về list các cụm tuổi/tháng.
    """
    if desc_all is None:
        return []

    pre = preprocess_text_for_age(desc_all)
    if not pre:
        return []

    full_text = pre.lower()
    text = full_text

    matches: List[str] = []
    used_spans: List[Tuple[int, int]] = []

    def overlap_span(start: int, end: int) -> bool:
        return any(not (end <= s or start >= e) for (s, e) in used_spans)

    def get_contexts(start_idx: int, end_idx: int) -> Tuple[str, str]:
        left_pos = max(0, start_idx - WINDOW_POS)
        right_pos = min(len(text), end_idx + WINDOW_POS)
        context_pos = text[left_pos:right_pos]

        left_neg = max(0, start_idx - WINDOW_NEG)
        right_neg = min(len(text), end_idx + WINDOW_NEG)
        context_neg = text[left_neg:right_neg]

        return context_pos, context_neg

    # 0) Range đặc biệt '3M-5T'
    for m in AGE_MT_RANGE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 1) Range "0-6 tháng ... đến ... 3 tuổi"
    for m in CROSS_MONTH_RANGE_TO_YEAR_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 2) Range newborn "sơ sinh ... đến ... 3 tuổi"
    for m in NEWBORN_RANGE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 3) Range 2 đơn vị: '6 tháng đến 5 tuổi', 'từ 1 tuần - 1 tuổi'
    for m in RANGE_BOTH_UNITS_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 4) Range 1 đơn vị: '0-12 tháng', '2-6 tuổi'
    for m in RANGE_ONE_UNIT_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 5) "độ tuổi từ 2 đến 6"
    for m in AGE_WORD_RANGE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 6) Single: 'dưới 12 tháng', 'từ 6 tháng tuổi', ...
    for m in SINGLE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue

        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context(context_neg):
            continue

        match_text = text[start_idx:end_idx].strip()

        # Nếu trước cụm này có 'độ tuổi phù hợp' ... => ép 'từ ...'
        ctx_left = text[max(0, start_idx - 60):start_idx]
        if any(kw in ctx_left for kw in [
            "độ tuổi phù hợp", "độ tuổi sử dụng", "độ tuổi khuyến nghị", "độ tuổi khuyên dùng"
        ]):
            if not re.search(r"\b(từ|trên|hơn|sau|dưới|under|over|more than)\b", match_text):
                match_text = "từ " + match_text

        matches.append(match_text)
        used_spans.append((start_idx, end_idx))

    return matches

In [13]:
# =========================================
# 6. CHUẨN HOÁ VỀ CANONICAL (THÁNG)
# =========================================

def _to_months(num_str: str, unit: str) -> int:
    """Chuyển '1', '1,5', '2.5' -> số tháng theo unit."""
    val = float(num_str.replace(",", "."))
    unit = unit.lower()

    if unit in MONTH_WORDS:
        return int(round(val))
    if unit in YEAR_WORDS:
        return int(round(val * 12))
    if unit in WEEK_WORDS:
        m = int(round(val * 7.0 / 30.0))
        return max(0, m)

    raise ValueError(f"Unit không hợp lệ: {unit}")


def normalize_age_phrase(raw: str) -> Optional[str]:
    """
    Chuẩn hóa 1 cụm tuổi/tháng về canonical theo THÁNG.

    Ưu tiên:
      - Các range ('A tháng đến B tuổi', '1-3 tuổi', '3m-5t')
      - Sau đó mới đến 'dưới N ...', 'từ N ...', 'N ... trở lên'
      - 'sơ sinh từ 1 tháng đến 2 tuổi' -> 1-24M (không phải 0-24M)
    """
    if not raw:
        return None

    s2 = raw.strip().lower()
    s2 = s2.replace("tháng tuổi", " tháng ")
    s2 = s2.replace("thang tuoi", " thang ")
    s2 = re.sub(r"\s+", " ", s2)

    # 0) '3m-5t'
    m_mt = re.search(
        r"(\d{1,2})\s*m\s*(?:-|–|—|đến|to)\s*(\d{1,2})\s*t",
        s2,
    )
    if m_mt:
        a_str, b_str = m_mt.group(1), m_mt.group(2)
        start_m = _to_months(a_str, "tháng")
        end_m   = _to_months(b_str, "tuổi")
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{start_m}-{end_m}M"

    # 1) RANGE 2 ĐƠN VỊ: '6 tháng đến 5 tuổi', 'từ 1 tháng đến 2 tuổi'
    m_both = re.search(
        r"(?:từ\s*)?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
        r"(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_both:
        a_str, u1, b_str, u2 = m_both.group(1), m_both.group(2), m_both.group(3), m_both.group(4)
        start_m = _to_months(a_str, u1)
        end_m   = _to_months(b_str, u2)
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{start_m}-{end_m}M"

    # 2) RANGE 1 ĐƠN VỊ: '1-3 tuổi', '0-12 tháng'
    m_range_unit = re.search(
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_range_unit:
        a_str = m_range_unit.group(1)
        b_str = m_range_unit.group(2)
        unit  = m_range_unit.group(3)
        start_m = _to_months(a_str, unit)
        end_m   = _to_months(b_str, unit)
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{start_m}-{end_m}M"

    # 3) 'độ tuổi từ 2 đến 6' (mặc định là năm)
    m_ageword = re.search(
        r"độ\s*tuổi[^0-9]{0,40}"
        r"(?:từ\s*)?"
        r"(\d{1,2}(?:[.,]\d+)?)\s*(?:-|–|—|đến|to)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)",
        s2,
    )
    if m_ageword:
        a_str = m_ageword.group(1)
        b_str = m_ageword.group(2)
        start_m = _to_months(a_str, "tuổi")
        end_m   = _to_months(b_str, "tuổi")
        if start_m > end_m:
            start_m, end_m = end_m, start_m
        return f"{start_m}-{end_m}M"

    # 4) XỬ LÝ CÁC CASE CÓ 'SƠ SINH'
    if "sơ sinh" in s2:
        # 4.1. Nếu có "từ X ... đến Y ..." -> ưu tiên X-Y (fix: "sơ sinh từ 1 tháng đến 2 tuổi")
        m_nb_range_xy = re.search(
            r"từ\s+(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years|y)\s*"
            r"(?:-|–|—|đến|to)\s*"
            r"(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years|y)",
            s2,
        )
        if m_nb_range_xy:
            a_str, u1, b_str, u2 = (
                m_nb_range_xy.group(1),
                m_nb_range_xy.group(2),
                m_nb_range_xy.group(3),
                m_nb_range_xy.group(4),
            )
            start_m = _to_months(a_str, u1)
            end_m   = _to_months(b_str, u2)
            if start_m > end_m:
                start_m, end_m = end_m, start_m
            return f"{start_m}-{end_m}M"

        # 4.2. 'sơ sinh ... đến ... N tháng/tuổi' -> 0-NM
        m_nb_range = re.search(
            r"(?:từ\s+giai\s*đoạn\s+)?sơ\s*sinh"
            r".{0,30}?(?:đến|tới|to|-).{0,30}?"
            r"(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years|y)",
            s2,
        )
        if m_nb_range:
            n_str = m_nb_range.group(1)
            unit  = m_nb_range.group(2)
            end_m = _to_months(n_str, unit)
            if end_m <= 0:
                return None
            return f"0-{end_m}M"

        # 4.3. 'từ sơ sinh' -> 0M+
        m_nb_from0 = re.search(r"từ\s+sơ\s*sinh\b(?:\s*(trở lên|\+))?", s2)
        if m_nb_from0:
            return "0M+"

        # 4.4. 'sơ sinh từ/hơn/trên N tháng/tuổi' -> NM+
        m_nb_plus = re.search(
            r"sơ\s*sinh\s*(từ|hơn|trên|sau|>=|over|more than)\s*"
            r"(\d{1,2}(?:[.,]\d+)?)\s*"
            r"(tháng|thang|month|months|tuổi|year|years|y)"
            r"(?:\s*(trở lên|\+))?",
            s2,
        )
        if m_nb_plus:
            n_str = m_nb_plus.group(2)
            unit  = m_nb_plus.group(3)
            start_m = _to_months(n_str, unit)
            return f"{start_m}M+"

    # 5) 'dưới N ...' -> '0-NM'
    m_under = re.search(
        r"(dưới|under|<)\s*(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)",
        s2,
    )
    if m_under:
        n_str = m_under.group(2)
        unit  = m_under.group(3)
        end_m = _to_months(n_str, unit)
        if end_m <= 0:
            return None
        return f"0-{end_m}M"

    # 6) 'từ / trên / hơn / sau N ...' -> 'NM+'
    m_from = re.search(
        r"(từ|trên|hơn|sau|>=|over|more than)\s*"
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)"
        r"(?:\s*(trở lên|\+))?",
        s2,
    )
    if m_from:
        n_str = m_from.group(2)
        unit  = m_from.group(3)
        start_m = _to_months(n_str, unit)
        return f"{start_m}M+"

    # 7) 'N ... trở lên' -> 'NM+'
    m_suffix_plus = re.search(
        r"(\d{1,2}(?:[.,]\d+)?)\s*"
        r"(tháng|thang|month|months|tuổi|year|years|tuần|tuan|week|weeks)\s*"
        r"(trở lên|\+)",
        s2,
    )
    if m_suffix_plus:
        n_str = m_suffix_plus.group(1)
        unit  = m_suffix_plus.group(2)
        start_m = _to_months(n_str, unit)
        return f"{start_m}M+"

    return None


def _parse_canonical(c: str) -> Tuple[int, Optional[int], bool]:
    """
    Trả (start_m, end_m, is_plus)
      - '6M'    -> (6, 6, False)
      - '6-36M' -> (6, 36, False)
      - '12M+'  -> (12, None, True)
    """
    if c.endswith("M+"):
        start = int(c[:-2])
        return start, None, True

    if not c.endswith("M"):
        raise ValueError(f"Canonical age phải kết thúc bằng 'M' hoặc 'M+': {c}")

    body = c[:-1]
    if "-" in body:
        a, b = body.split("-", 1)
        return int(a), int(b), False
    else:
        v = int(body)
        return v, v, False


def normalize_age_phrase_list(raw_list: Optional[List[str]]) -> List[str]:
    if raw_list is None:
        return []

    canon: List[str] = []
    for raw in raw_list:
        norm = normalize_age_phrase(raw)
        if norm is not None:
            canon.append(norm)

    # unique theo thứ tự
    uniq: List[str] = []
    for c in canon:
        if c not in uniq:
            uniq.append(c)

    if not uniq:
        return []

    parsed = [_parse_canonical(c) for c in uniq]
    keep = [True] * len(uniq)

    for i, (si, ei, plus_i) in enumerate(parsed):
        if not keep[i]:
            continue
        for j, (sj, ej, plus_j) in enumerate(parsed):
            if i == j:
                continue

            # j bao phủ i?
            if plus_j:
                # j là 'sjM+' -> [sj, +∞)
                if not plus_i:
                    continue
                # cả 2 là '+': giữ cái xuất hiện trước
                if ej is None and j < i:
                    keep[i] = False
                    break
            else:
                # j là khoảng hữu hạn [sj, ej]
                if ei is None:
                    # khoảng vô hạn không bị cover bởi hữu hạn
                    continue
                if sj <= si and ej >= ei:
                    keep[i] = False
                    break

    return [c for c, k in zip(uniq, keep) if k]

In [14]:
# df_unknown_desc đã được bạn tạo từ trước, gồm:
# item_id, description, description_new, age_group_final
# với điều kiện:
#   - age_group_final == "Không xác định"
#   - ít nhất 1 trong 2 description có thông tin

df_unknown = (
    df_unknown_desc
    .with_columns(
        pl.concat_str(
            [
                pl.col("description").fill_null(""),
                pl.col("description_new").fill_null(""),
            ],
            separator=" "
        ).alias("desc_all")
    )
    .with_columns(
        pl.col("desc_all").map_elements(
            extract_age_phrases,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_phrases_raw")
    )
    .with_columns(
        pl.col("age_phrases_raw").map_elements(
            normalize_age_phrase_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_ranges_norm")
    )
)

# Lấy tất cả canonical ranges đã detect
target_norm_list = (
    df_unknown
    .select(pl.col("age_ranges_norm"))
    .explode("age_ranges_norm")
    .filter(pl.col("age_ranges_norm").is_not_null())
    .unique()
    .to_series()
    .to_list()
)

print("Số lượng target_norm:", len(target_norm_list))
print(target_norm_list)

# Xuất ví dụ để review
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(50)

def wrap175(text):
    if text is None:
        return ""
    return textwrap.fill(str(text), width=175)

with open("output_age_norm.txt", "w", encoding="utf-8") as f:

    for tg in target_norm_list:
        f.write(f"\n=== VÍ DỤ CHO TARGET_NORM = {tg} ===\n\n")
        target_norm = tg

        examples = (
            df_unknown
            .filter(pl.col("age_ranges_norm").list.contains(target_norm))
            .select([
                "item_id",
                "age_group_final",
                "age_phrases_raw",
                "age_ranges_norm",
                "description",
                "description_new",
            ])
            .head(20)
        )

        for row in examples.iter_rows(named=True):
            f.write("======================================\n")
            f.write(f"item_id: {row['item_id']}\n")
            f.write(f"age_group_final: {row['age_group_final']}\n")
            f.write(f"age_phrases_raw: {row['age_phrases_raw']}\n")
            f.write(f"age_ranges_norm: {row['age_ranges_norm']}\n\n")

            f.write("description:\n")
            f.write(wrap175(row["description"]) + "\n\n")

            f.write("description_new:\n")
            f.write(wrap175(row["description_new"]) + "\n\n")

print("Đã xuất file: output_age_norm.txt")


Số lượng target_norm: 57
['24-96M', '7-24M', '12M+', '9M+', '24-72M', '6-12M', '48-60M', '36M+', '10M+', '15M+', '0M+', '3-24M', '3-36M', '0-3M', '84-144M', '0-84M', '0-36M', '7M+', '0-72M', '4M+', '3-18M', '12-36M', '60M+', '48-72M', '36-48M', '3-60M', '0-12M', '3M+', '216M+', '18M+', '228M+', '1-24M', '12-72M', '6M+', '216-360M', '36-144M', '16M+', '0-144M', '24M+', '144M+', '3-6M', '1M+', '9-24M', '36-120M', '216-720M', '0-48M', '3-12M', '24-36M', '96M+', '0-6M', '0-10M', '36-96M', '8M+', '0-24M', '48M+', '6-11M', '72M+']
Đã xuất file: output_age_norm.txt


##### Fill các giá trị bắt được từ target_norm

In [15]:
import polars as pl

# df_unknown hiện đang có:
# - item_id
# - age_ranges_norm: List[str] (canonical, ví dụ: ["0-12M", "12-36M", "60M+"])

# 1) explode để mỗi dòng là 1 (item_id, age_range)
df_item_norm = (
    df_unknown
    .select(["item_id", "age_ranges_norm"])
    .explode("age_ranges_norm")
    .filter(pl.col("age_ranges_norm").is_not_null())
)

# 2) định nghĩa hàm chuyển "XM+" -> "Từ XM"
def convert_plus_list(lst: list[str]) -> list[str]:
    out = []
    for s in lst:
        if s.endswith("M+"):
            num = s[:-2]           # "60M+" -> "60"
            out.append(f"Từ {num}M")
        else:
            out.append(s)
    return out

# 3) group_by theo item_id, unique + sort, rồi apply convert_plus_list
df_item_age_dict = (
    df_item_norm
    .group_by("item_id")
    .agg(
        pl.col("age_ranges_norm").unique()  # list các canonical khác nhau
    )
    .with_columns(
        pl.col("age_ranges_norm").map_elements(
            convert_plus_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_dict_list")
    )
    # tạo thêm bản string dạng ["0-12M","Từ 12M"] để fill vào age_group_final
    .with_columns(
        pl.col("age_dict_list").map_elements(
            lambda lst: "[" + ", ".join(f'"{x}"' for x in lst) + "]",
            return_dtype=pl.Utf8
        ).alias("age_dict_str")
    )
)

# df_item_age_dict hiện có:
# - item_id
# - age_ranges_norm (list canonical gốc)
# - age_dict_list (list đã đổi M+ -> "Từ XM")
# - age_dict_str  (string để fill vào age_group_final)


In [16]:
# Tạo dictionary dùng cho việc debug / dùng ngoài Polars
item_to_age_dict = {
    row["item_id"]: row["age_dict_list"]
    for row in df_item_age_dict.select(["item_id", "age_dict_list"]).to_dicts()
}

# Giờ item_to_age_dict[item_id] sẽ là list, ví dụ:
# ["0-12M", "12-36M", "Từ 60M"]


In [17]:
# Join mapping theo item_id, chỉ fill cho các dòng "Không xác định"

df_age_filled = (
    df_age
    .join(
        df_item_age_dict.select(["item_id", "age_dict_str"]),
        on="item_id",
        how="left"
    )
    .with_columns(
        pl.when(
            (pl.col("age_group_final") == "Không xác định")
            & pl.col("age_dict_str").is_not_null()
        )
        .then(pl.col("age_dict_str"))         # dùng dictionary dạng ["...","..."]
        .otherwise(pl.col("age_group_final"))
        .alias("age_group_final_filled")
    )
    .drop("age_group_final", "age_dict_str")
    .rename({"age_group_final_filled": "age_group_final"})
)

# df_age_filled là dataframe cuối cùng sau khi fill.


In [18]:
import polars as pl

# df_unknown: item_id, age_ranges_norm

df_item_norm = (
    df_unknown
    .select(["item_id", "age_ranges_norm"])
    .explode("age_ranges_norm")
    .filter(pl.col("age_ranges_norm").is_not_null())
)

def convert_plus_list(lst):
    # lst có thể là list hoặc Series
    if isinstance(lst, pl.Series):
        lst = lst.to_list()

    out = []
    for s in lst:
        if not isinstance(s, str):
            out.append(s)
            continue

        s_clean = s.strip()

        # CASE 1: dạng "XM+"
        if s_clean.endswith("M+"):
            num = s_clean[:-2]
            out.append(f"Từ {num}M")
            continue

        # CASE 2: dạng chữ "trên 6M" / "Trên 6M" / "hơn 12M" / "Hơn 12M"
        m = re.match(r"(?i)^(trên|hơn)\s*(\d+)\s*m$", s_clean)
        if m:
            num = m.group(2)
            out.append(f"Từ {num}M")
            continue

        # giữ nguyên các dạng khác ("0-6M", "6-12M", ...)
        out.append(s_clean)

    return out

def build_age_fill_str(lst):
    if isinstance(lst, pl.Series):
        lst = lst.to_list()
    if lst is None or len(lst) == 0:
        return None
    if len(lst) == 1:
        # trả về string đơn
        return lst[0]
    # nhiều giá trị → JSON-like list với dấu "
    return "[" + ", ".join(f"\"{x}\"" for x in lst) + "]"

df_item_age_dict = (
    df_item_norm
    .group_by("item_id")
    .agg(pl.col("age_ranges_norm").unique().sort())
    .with_columns(
        pl.col("age_ranges_norm").map_elements(
            convert_plus_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("age_dict_list")
    )
    .with_columns(
        pl.col("age_dict_list").map_elements(
            build_age_fill_str,
            return_dtype=pl.Utf8
        ).alias("age_fill_str")
    )
)

In [19]:
# Thống kê trước khi fill
total_rows = df_age.height
unknown_before = df_age.filter(pl.col("age_group_final") == "Không xác định").height
ratio_before = unknown_before / total_rows * 100

print("=== TRƯỚC KHI FILL ===")
print(f"Số dòng 'Không xác định': {unknown_before}")
print(f"Tỉ lệ: {ratio_before:.2f}%")
print()


# Fill
df_age_filled = (
    df_age
    .with_columns(
        pl.col("age_group_final").alias("age_group_final_before")
    )
    .join(
        df_item_age_dict.select(["item_id", "age_fill_str"]),
        on="item_id",
        how="left"
    )
    .with_columns(
        pl.when(
            (pl.col("age_group_final_before") == "Không xác định")
            & pl.col("age_fill_str").is_not_null()
        )
        .then(pl.col("age_fill_str"))
        .otherwise(pl.col("age_group_final_before"))
        .alias("age_group_final")
    )
    .drop("age_fill_str")
)

# KHÔNG đổi dấu nháy nữa. Không .str.replace() gì thêm.


# Thống kê sau fill
unknown_after = df_age_filled.filter(pl.col("age_group_final") == "Không xác định").height
ratio_after = unknown_after / total_rows * 100

print("=== SAU KHI FILL ===")
print(f"Số dòng 'Không xác định': {unknown_after}")
print(f"Tỉ lệ: {ratio_after:.2f}%")
print()


# Lấy ra các dòng đã fill
df_filled_rows = (
    df_age_filled
    .filter(
        (pl.col("age_group_final_before") == "Không xác định")
        & (pl.col("age_group_final") != "Không xác định")
    )
    .select([
        "item_id",
        "description",
        "description_new",
        "age_group_final_before",
        "age_group_final",
    ])
)

pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_rows(100)

print("=== CÁC DÒNG ĐÃ FILL ===")
print(df_filled_rows)

=== TRƯỚC KHI FILL ===
Số dòng 'Không xác định': 10857
Tỉ lệ: 39.74%

=== SAU KHI FILL ===
Số dòng 'Không xác định': 10595
Tỉ lệ: 38.78%

=== CÁC DÒNG ĐÃ FILL ===
shape: (262, 5)
┌───────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│ item_id       ┆ description        ┆ description_new    ┆ age_group_final_be ┆ age_group_final   │
│ ---           ┆ ---                ┆ ---                ┆ fore               ┆ ---               │
│ str           ┆ str                ┆ str                ┆ ---                ┆ str               │
│               ┆                    ┆                    ┆ str                ┆                   │
╞═══════════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════════╡
│ 0024180250008 ┆ Quần kaki bé trai  ┆ Không xác định     ┆ Không xác định     ┆ 3-60M             │
│               ┆ ngắn Laluna Quần   ┆                    ┆                    ┆                   │
│            

In [20]:
split_and_save_parquet(df_age_filled, 1, "./preprocessed-dataset")

Đã lưu file: ./preprocessed-dataset/sale_pers.item_chunk_0.parquet
